# IQA Healthcheck — DataScout Environments

Probes every IQA across all environments and fetches 5 sample rows as evidence.

| Status | Meaning |
|---|---|
| ✅ Working | HTTP 200, rows returned |
| ⚠️ Empty | HTTP 200, TotalCount = 0 (deployed but no data yet) |
| ⚠️ Params | HTTP 400 — IQA requires parameters to run |
| ❌ Broken | HTTP 4xx/5xx (not 400) or timeout |
| ❌ Missing | IQA path not found in this environment |

**Golden record:** `demo83`

In [14]:
%pip install python-dotenv onepassword-sdk requests pandas --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import sys, os, time, requests
import pandas as pd
from pathlib import Path
from IPython.display import display

repo_root = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(repo_root))

from common.onepassword_manager import OnePasswordManager
from common.imis_client import IMISClient

op = OnePasswordManager()
print("✅ Modules loaded.")

✅ Modules loaded.


In [16]:
GOLDEN_RECORD   = "demo83"
IQA_ROOT        = "$/_DataScout"
DEMO_EXCLUDE    = "$/_DataScout/Demo"
SAMPLE_LIMIT    = 5
REQUEST_TIMEOUT = 15  # seconds

CLIENT_IDS = [
    "demo83",
    "aaae", "atsdemo89", "aboncle", "armdemo96", "imisdemo11", "imis36",
    "apimisdemo25", "imis87", "imis104", "demosales3",
    "atdemo2", "atdemo81", "demo42", "bsidemo27", "ccs", "demo86",
    "ensyncdemo13", "i8vdemo13", "ibcdemo80", "isgdemo14", "isgdemo106",
    "oasw",
]

STATUS_ICON = {
    "working":      "✅ Working",
    "empty":        "⚠️ Empty",
    "params":       "⚠️ Params",
    "broken":       "❌ Broken",
    "missing":      "❌ Missing",
    "login_failed": "❌ Login Failed",
}

print(f"Golden record : {GOLDEN_RECORD}")
print(f"Environments  : {len(CLIENT_IDS)}")

Golden record : demo83
Environments  : 23


In [17]:
# ── Fetch credentials from 1Password ──────────────────────────────────────
print("Fetching credentials from 1Password...\n")
environment_credentials = {}

for client_id in CLIENT_IDS:
    try:
        secrets  = await op.get_flattened_client_item(client_id)
        base_url = secrets.get("imis_base_url")
        username = secrets.get("username")
        password = secrets.get("password")
        if not all([base_url, username, password]):
            print(f"  SKIP  {client_id}: missing field(s). Found: {list(secrets.keys())}")
            continue
        environment_credentials[client_id] = {
            "imis_base_url": base_url,
            "username":      username,
            "password":      password,
        }
        print(f"  OK    {client_id}  ({base_url})")
    except Exception as e:
        print(f"  FAIL  {client_id}: {e}")

# ── URL overrides (correct 1Password entries here until updated at source) ──
URL_OVERRIDES = {
    "cpanb": "https://staff.cpanewbrunswick.ca",
}
for env, url in URL_OVERRIDES.items():
    if env in environment_credentials:
        print(f"  OVERRIDE  {env}  →  {url}")
        environment_credentials[env]["imis_base_url"] = url

print(f"\nLoaded {len(environment_credentials)}/{len(CLIENT_IDS)} environments.")

Fetching credentials from 1Password...

  OK    demo83  (https://demoaisp83.imiscloud.com/)
  OK    aaae  (https://member.aaae.org/)
  OK    atsdemo89  (https://demoaisp89.imiscloud.com/)
  OK    aboncle  (https://abo-ncle.org/)
  OK    armdemo96  (https://demoaisp96.imiscloud.com/)
  OK    imisdemo11  (https://demosales11.imiscloud.com/)
  OK    imis36  (https://demosales36.imiscloud.com/)
  OK    apimisdemo25  (https://apdemosales25.imiscloud.com/)
  OK    imis87  (https://demosales87.imiscloud.com/)
  OK    imis104  (https://demosales104.imiscloud.com/)
  OK    demosales3  (https://demosales3.imiscloud.com/)
  OK    atdemo2  (https://demoaisp2.imiscloud.com/)
  OK    atdemo81  (https://demoaisp81.imiscloud.com/)
  OK    demo42  (https://demoaisp42.imiscloud.com/)
  OK    bsidemo27  (https://demoaisp27.imiscloud.com/)
  OK    ccs  (https://my.ccs.ca/)
  OK    demo86  (https://demoaisp86.imiscloud.com/)
  OK    ensyncdemo13  (https://demoaisp13.imiscloud.com/)
  OK    i8vdemo13  (http

In [18]:
# ── Discover IQAs from golden record ──────────────────────────────────────
IQA_EXCLUDE_PATHS = {
    "$/_DataScout/TestJamesQuery",
}

print(f"Connecting to golden record: {GOLDEN_RECORD}...")
g = environment_credentials[GOLDEN_RECORD]
golden_client = IMISClient(g["imis_base_url"], g["username"], g["password"])

all_docs = golden_client.list_iqas_in_folder(IQA_ROOT)
golden_iqas = [
    doc for doc in all_docs
    if doc.get("Type") == "IQD"
    and not (doc.get("Path") or "").startswith(DEMO_EXCLUDE)
    and (doc.get("Path") or "") not in IQA_EXCLUDE_PATHS
]
golden_paths = [doc["Path"] for doc in golden_iqas]

print(f"\n✅ {len(golden_paths)} IQAs in golden record:")
for p in golden_paths:
    print(f"  {p}")

Connecting to golden record: demo83...
ℹ️[IMISClient] Token expires at 2026-03-30 16:00:27.625186
🟢[IMISClient] Token successfully retrieved

✅ 30 IQAs in golden record:
  $/_DataScout/Triggers/changelog_by_date
  $/_DataScout/AiParts/Profile/Datascout AiParts - Profile Security
  $/_DataScout/Segments/tags_added_by_date
  $/_DataScout/AiParts/Profile/association_average_revenue
  $/_DataScout/Segments/tags_deleted_by_date
  $/_DataScout/Segments/tag_refresh_name_log
  $/_DataScout/AiParts/Profile/sso_retrieve_data_by_email
  $/_DataScout/AiParts/Profile/properties_by_id
  $/_DataScout/Triggers/unsubscribe
  $/_DataScout/Triggers/communication_preferences
  $/_DataScout/AiParts/Profile/profile_completeness_by_id
  $/_DataScout/AiParts/Profile/member_type_lookup
  $/_DataScout/AiParts/Profile/event_registrations_by_id
  $/_DataScout/Segments/tag_refresh_tenure
  $/_DataScout/Segments/tag_refresh_age
  $/_DataScout/Segments/tag_refresh_change_log
  $/_DataScout/Triggers/resubscribe
  $/_

In [19]:
# ── Core probe function ────────────────────────────────────────────────────
import datetime as dt

def probe_iqa(client: IMISClient, path: str, limit: int = 5, timeout: int = 15) -> dict:
    """
    Raw GET /api/IQA — captures HTTP status and sample rows.
    Returns: status, http_status, total_count, sample_rows, error_msg, duration_sec
    """
    # Ensure token is fresh before bypassing make_request
    if client.token_expiration is None or dt.datetime.now() >= client.token_expiration:
        client.authenticate()

    url     = f"{client.base_url}/api/IQA"
    params  = {"QueryName": path, "limit": limit, "offset": 0}
    headers = {"Authorization": client.token}

    t0 = time.time()
    try:
        resp     = requests.get(url, params=params, headers=headers, timeout=timeout)
        duration = round(time.time() - t0, 2)

        if resp.status_code == 200:
            data  = resp.json()
            total = data.get("TotalCount", 0)

            # Flatten sample rows (same logic as IMISClient._simplify_data)
            sample = []
            for row in data.get("Items", {}).get("$values", [])[:limit]:
                rec = {}
                for kv in row.get("Properties", {}).get("$values", []):
                    v = kv.get("Value")
                    rec[kv["Name"]] = v.get("$value", v) if isinstance(v, dict) else v
                sample.append(rec)

            return {
                "status":       "working" if total > 0 else "empty",
                "http_status":  200,
                "total_count":  total,
                "sample_rows":  sample,
                "error_msg":    None,
                "duration_sec": duration,
            }

        elif resp.status_code == 400:
            try:    msg = resp.json().get("Message") or resp.text[:300]
            except: msg = resp.text[:300]
            return {
                "status": "params", "http_status": 400,
                "total_count": 0, "sample_rows": [],
                "error_msg": msg, "duration_sec": duration,
            }

        else:
            try:    msg = resp.json().get("Message") or resp.text[:300]
            except: msg = resp.text[:300]
            return {
                "status": "broken", "http_status": resp.status_code,
                "total_count": 0, "sample_rows": [],
                "error_msg": msg, "duration_sec": duration,
            }

    except requests.exceptions.Timeout:
        return {
            "status": "broken", "http_status": None,
            "total_count": 0, "sample_rows": [],
            "error_msg": f"Timeout after {timeout}s",
            "duration_sec": round(time.time() - t0, 2),
        }
    except Exception as e:
        return {
            "status": "broken", "http_status": None,
            "total_count": 0, "sample_rows": [],
            "error_msg": str(e),
            "duration_sec": round(time.time() - t0, 2),
        }

print("✅ probe_iqa() ready.")

✅ probe_iqa() ready.


In [20]:
# ── Run healthcheck across all environments ────────────────────────────────
global_start = time.time()
all_results  = []

for client_id, creds in environment_credentials.items():
    print(f"\n{'─'*60}")
    print(f"  {client_id}")
    print(f"{'─'*60}")

    # Login
    try:
        imis = IMISClient(creds["imis_base_url"], creds["username"], creds["password"])
    except Exception as e:
        print(f"  ❌ Login failed: {e}")
        for path in golden_paths:
            all_results.append({
                "env": client_id, "path": path,
                "status": "login_failed", "http_status": None,
                "total_count": None, "sample_rows": [],
                "error_msg": str(e), "duration_sec": 0,
            })
        continue

    # Discover which IQA paths exist in this env
    try:
        env_docs   = imis.list_iqas_in_folder(IQA_ROOT)
        env_paths  = {
            doc["Path"] for doc in env_docs
            if doc.get("Type") == "IQD"
            and not (doc.get("Path") or "").startswith(DEMO_EXCLUDE)
        }
    except Exception as e:
        print(f"  ⚠️  IQA discovery failed: {e}")
        env_paths = set()

    # Probe each golden IQA
    for path in golden_paths:
        if path not in env_paths:
            result = {
                "status": "missing", "http_status": None,
                "total_count": 0, "sample_rows": [],
                "error_msg": "Not in document tree", "duration_sec": 0,
            }
        else:
            result = probe_iqa(imis, path, limit=SAMPLE_LIMIT, timeout=REQUEST_TIMEOUT)

        icon = STATUS_ICON[result["status"]]
        print(f"  {icon:<18} {path}  (HTTP={result['http_status']}, rows={result['total_count']})")

        all_results.append({"env": client_id, "path": path, **result})

global_duration = round(time.time() - global_start, 2)
print(f"\n\n⏱️  Completed in {global_duration}s across {len(environment_credentials)} environments.")


────────────────────────────────────────────────────────────
  demo83
────────────────────────────────────────────────────────────
ℹ️[IMISClient] Token expires at 2026-03-30 16:00:28.785504
🟢[IMISClient] Token successfully retrieved
  ⚠️ Params          $/_DataScout/Triggers/changelog_by_date  (HTTP=400, rows=0)
  ✅ Working          $/_DataScout/AiParts/Profile/Datascout AiParts - Profile Security  (HTTP=200, rows=1)
  ✅ Working          $/_DataScout/Segments/tags_added_by_date  (HTTP=200, rows=49993)
  ✅ Working          $/_DataScout/AiParts/Profile/association_average_revenue  (HTTP=200, rows=1)
  ✅ Working          $/_DataScout/Segments/tags_deleted_by_date  (HTTP=200, rows=18687)
  ✅ Working          $/_DataScout/Segments/tag_refresh_name_log  (HTTP=200, rows=1656)
  ⚠️ Params          $/_DataScout/AiParts/Profile/sso_retrieve_data_by_email  (HTTP=400, rows=0)
  ✅ Working          $/_DataScout/AiParts/Profile/properties_by_id  (HTTP=200, rows=64)
  ✅ Working          $/_DataScout/

In [21]:
# ── Build results DataFrame ────────────────────────────────────────────────
df = pd.DataFrame(all_results)
print(f"✅ {len(df)} probes complete — {df['env'].nunique()} environments × {df['path'].nunique()} IQAs.")

✅ 690 probes complete — 23 environments × 30 IQAs.


In [22]:
# ── 🔴 SECTION 1: Environments Down ───────────────────────────────────────
print("=" * 65)
print("  🔴  SECTION 1 — ENVIRONMENTS DOWN")
print("=" * 65)

# Login failed — can't reach the API at all
login_failed_envs = (
    df[df["status"] == "login_failed"]
    .drop_duplicates("env")[["env", "error_msg"]]
    .copy()
)
login_failed_envs["url"] = login_failed_envs["env"].map(
    lambda e: environment_credentials.get(e, {}).get("imis_base_url", "")
)

# Not deployed — login worked but zero IQAs found
not_deployed_envs = (
    df.groupby("env")["status"]
    .apply(lambda s: (s == "missing").all())
    .pipe(lambda s: s[s].index.tolist())
)
not_deployed_envs = [e for e in not_deployed_envs if e not in login_failed_envs["env"].values]

# ── Print ──
if login_failed_envs.empty and not not_deployed_envs:
    print("\n  ✅  All environments reachable and DataScout deployed.\n")
else:
    if not login_failed_envs.empty:
        print(f"\n  ❌  Login Failed ({len(login_failed_envs)} env{'s' if len(login_failed_envs) > 1 else ''})\n")
        for _, row in login_failed_envs.iterrows():
            print(f"      • {row['env']:<20}  {row['url']}")
            print(f"        {row['error_msg']}")

    if not_deployed_envs:
        print(f"\n  ❌  DataScout Not Deployed ({len(not_deployed_envs)} env{'s' if len(not_deployed_envs) > 1 else ''})\n")
        for env in not_deployed_envs:
            url = environment_credentials.get(env, {}).get("imis_base_url", "")
            print(f"      • {env:<20}  {url}")

  🔴  SECTION 1 — ENVIRONMENTS DOWN

  ❌  Login Failed (1 env)

      • ccs                   https://my.ccs.ca/
        🔴[IMISClient] Failed to authenticate with the API


In [23]:
# ── 🔴 SECTION 2: Broken IQAs ─────────────────────────────────────────────
print("=" * 65)
print("  🔴  SECTION 2 — BROKEN IQAs")
print("=" * 65)

broken = df[df["status"] == "broken"].copy()
broken["iqa"] = broken["path"].str.replace("$/_DataScout/", "", regex=False)

if broken.empty:
    print("\n  ✅  No broken IQAs.\n")
else:
    for iqa, group in broken.groupby("iqa"):
        n = len(group)
        print(f"\n  ❌  {iqa}  ({n} env{'s' if n > 1 else ''})")
        for _, row in group.iterrows():
            print(f"      • {row['env']:<20}  HTTP {row['http_status']}  —  {row['error_msg']}")

  🔴  SECTION 2 — BROKEN IQAs

  ❌  AiParts/Profile/contact_by_id  (2 envs)
      • aaae                  HTTP nan  —  Timeout after 15s
      • aboncle               HTTP nan  —  Timeout after 15s


In [24]:
# ── 🟡 SECTION 3: IQA Issues ──────────────────────────────────────────────
print("=" * 65)
print("  🟡  SECTION 3 — IQA ISSUES")
print("=" * 65)

# Envs to skip in this section (already covered in Section 1)
ignore_envs = set(login_failed_envs["env"].tolist() + not_deployed_envs)

# IQAs that are expected to need params (params in the golden record itself)
golden_statuses = df[df["env"] == GOLDEN_RECORD].set_index("path")["status"]
expected_params_paths = set(golden_statuses[golden_statuses == "params"].index)

# ── Unexpected Params: IQA returns 400 here but works elsewhere ──
unexpected_params = df[
    (df["status"] == "params")
    & (~df["path"].isin(expected_params_paths))
    & (~df["env"].isin(ignore_envs))
].copy()
unexpected_params["iqa"] = unexpected_params["path"].str.replace("$/_DataScout/", "", regex=False)

print("\n  ── Unexpected ⚠️  Params (IQA works elsewhere but requires params here) ──\n")
if unexpected_params.empty:
    print("  ✅  None.\n")
else:
    for iqa, group in unexpected_params.groupby("iqa"):
        envs = ", ".join(sorted(group["env"].tolist()))
        print(f"  ⚠️   {iqa}")
        print(f"       Affected: {envs}\n")

# ── Missing: IQA deployed elsewhere but absent here ──
missing = df[
    (df["status"] == "missing")
    & (~df["env"].isin(ignore_envs))
].copy()
missing["iqa"] = missing["path"].str.replace("$/_DataScout/", "", regex=False)

print("  ── ❌  Missing IQAs (deployed elsewhere, absent here) ──\n")
if missing.empty:
    print("  ✅  None.\n")
else:
    for iqa, group in missing.groupby("iqa"):
        envs = ", ".join(sorted(group["env"].tolist()))
        n = len(group)
        print(f"  ❌  {iqa}  ({n} env{'s' if n > 1 else ''})")
        print(f"       Affected: {envs}\n")

  🟡  SECTION 3 — IQA ISSUES

  ── Unexpected ⚠️  Params (IQA works elsewhere but requires params here) ──

  ⚠️   AiParts/Profile/association_average_revenue
       Affected: aboncle

  ── ❌  Missing IQAs (deployed elsewhere, absent here) ──

  ❌  Segments/all_member_ids  (1 env)
       Affected: demo42

  ❌  Segments/contact_export  (1 env)
       Affected: demo42

  ❌  Triggers/communication_preferences  (1 env)
       Affected: aaae

  ❌  Triggers/resubscribe  (1 env)
       Affected: aaae



In [25]:
# ── Sample Evidence (optional) ─────────────────────────────────────────────
# Set to True to inspect raw row samples per IQA per environment
SHOW_SAMPLES = False

if SHOW_SAMPLES:
    working_df = df[df["status"] == "working"]
    for path in golden_paths:
        rows = working_df[working_df["path"] == path]
        if rows.empty:
            continue
        print(f"\n{'='*70}")
        print(f"  {path}")
        print(f"{'='*70}")
        for _, row in rows.iterrows():
            samples = row["sample_rows"]
            if not samples:
                continue
            print(f"\n  [{row['env']}]  {len(samples)} rows  (TotalCount={row['total_count']})")
            display(pd.DataFrame(samples))

In [26]:
# ── HTML Report ────────────────────────────────────────────────────────────
import datetime as _dt
import math as _math
import webbrowser
from pathlib import Path

def _build_report(now_str):
    RED    = "#f85149"
    YELLOW = "#e3b341"
    n_envs = df["env"].nunique()
    n_iqas = df["path"].nunique()

    def _http(val):
        return "—" if val is None or (isinstance(val, float) and _math.isnan(val)) else int(val)

    def card(content):
        return f'<div class="iqa-block">{content}</div>'

    def section(icon, title, color, body):
        return f"""
        <div class="section">
          <div class="section-header" style="border-left:3px solid {color}">
            <span class="section-title">{icon}&nbsp; {title}</span>
          </div>
          <div class="section-body">{body}</div>
        </div>"""

    def table(headers, rows_html):
        ths = "".join(f"<th>{h}</th>" for h in headers)
        return f"<table><thead><tr>{ths}</tr></thead><tbody>{rows_html}</tbody></table>"

    def ok(msg):
        return f'<p class="ok">✅&nbsp; {msg}</p>'

    # ── Section 1 ──────────────────────────────────────────────────────────
    s1 = ""
    if login_failed_envs.empty and not not_deployed_envs:
        s1 = ok("All environments reachable and DataScout deployed.")
    else:
        if not login_failed_envs.empty:
            rows = "".join(
                f"<tr><td class='env'>{r['env']}</td>"
                f"<td class='mono dim'>{r['url']}</td>"
                f"<td class='mono red'>{r['error_msg']}</td></tr>"
                for _, r in login_failed_envs.iterrows()
            )
            s1 += f"""
            <div class="sub">
              <div class="sub-title">❌ Login Failed <span class="badge">{len(login_failed_envs)}</span></div>
              {table(["Env", "URL", "Error"], rows)}
            </div>"""
        if not_deployed_envs:
            rows = "".join(
                f"<tr><td class='env'>{env}</td>"
                f"<td class='mono dim'>{environment_credentials.get(env,{}).get('imis_base_url','')}</td></tr>"
                for env in not_deployed_envs
            )
            s1 += f"""
            <div class="sub">
              <div class="sub-title">❌ DataScout Not Deployed <span class="badge">{len(not_deployed_envs)}</span></div>
              {table(["Env", "URL"], rows)}
            </div>"""

    # ── Section 2 ──────────────────────────────────────────────────────────
    if broken.empty:
        s2 = ok("No broken IQAs.")
    else:
        s2 = ""
        for iqa, grp in broken.groupby("iqa"):
            rows = "".join(
                f"<tr><td class='env'>{r['env']}</td>"
                f"<td class='mono dim'>HTTP {_http(r['http_status'])}</td>"
                f"<td class='mono red'>{r['error_msg']}</td></tr>"
                for _, r in grp.iterrows()
            )
            s2 += card(
                f"<div class='iqa-name red'>❌ {iqa} <span class='badge'>{len(grp)} env{'s' if len(grp)>1 else ''}</span></div>"
                + table(["Env", "HTTP", "Error"], rows)
            )

    # ── Section 3 ──────────────────────────────────────────────────────────
    s3 = ""
    s3 += '<div class="sub-title" style="margin-bottom:12px">⚠️ Unexpected Params <span class="sub-label">— IQA works elsewhere but requires params here</span></div>'
    if unexpected_params.empty:
        s3 += ok("None.")
    else:
        for iqa, grp in unexpected_params.groupby("iqa"):
            envs = ", ".join(sorted(grp["env"].tolist()))
            s3 += card(
                f"<div class='iqa-name yellow'>⚠️ {iqa}</div>"
                f"<div class='affected'>Affected: {envs}</div>"
            )

    s3 += '<div class="sub-title" style="margin-top:24px;margin-bottom:12px">❌ Missing IQAs <span class="sub-label">— deployed elsewhere, absent here</span></div>'
    if missing.empty:
        s3 += ok("None.")
    else:
        for iqa, grp in missing.groupby("iqa"):
            envs = ", ".join(sorted(grp["env"].tolist()))
            s3 += card(
                f"<div class='iqa-name red'>❌ {iqa} <span class='badge'>{len(grp)} env{'s' if len(grp)>1 else ''}</span></div>"
                f"<div class='affected'>Affected: {envs}</div>"
            )

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>DataScout IQA Healthcheck — {now_str}</title>
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{background:#0d1117;color:#e6edf3;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;font-size:14px;line-height:1.6;padding:40px 32px;max-width:1000px;margin:0 auto}}
  header{{margin-bottom:40px;padding-bottom:20px;border-bottom:1px solid #21262d}}
  header h1{{font-size:20px;font-weight:700;margin-bottom:6px;letter-spacing:-0.3px}}
  .meta{{color:#8b949e;font-size:13px}}
  .section{{margin-bottom:28px;border:1px solid #21262d;border-radius:10px;overflow:hidden;background:#161b22}}
  .section-header{{padding:14px 20px;background:#161b22;border-bottom:1px solid #21262d}}
  .section-title{{font-size:14px;font-weight:600;letter-spacing:0.2px}}
  .section-body{{padding:20px}}
  .sub{{margin-bottom:22px}}
  .sub:last-child{{margin-bottom:0}}
  .sub-title{{font-size:13px;font-weight:600;color:#c9d1d9;margin-bottom:10px}}
  .sub-label{{font-weight:400;font-size:12px;color:#6e7681}}
  table{{width:100%;border-collapse:collapse;font-size:13px;margin-top:8px}}
  th{{text-align:left;padding:7px 12px;color:#8b949e;font-weight:500;font-size:11px;text-transform:uppercase;letter-spacing:0.5px;border-bottom:1px solid #21262d}}
  td{{padding:9px 12px;border-bottom:1px solid #21262d;vertical-align:top}}
  tr:last-child td{{border-bottom:none}}
  td.env{{font-weight:600;color:#58a6ff;white-space:nowrap}}
  td.mono{{font-family:"SF Mono",Menlo,monospace;font-size:12px}}
  td.dim{{color:#8b949e}}
  td.red{{color:#f85149}}
  .iqa-block{{background:#0d1117;border:1px solid #21262d;border-radius:6px;padding:14px 16px;margin-bottom:10px}}
  .iqa-block:last-child{{margin-bottom:0}}
  .iqa-name{{font-size:13px;font-weight:600;margin-bottom:6px}}
  .iqa-name.red{{color:#f85149}}
  .iqa-name.yellow{{color:#e3b341}}
  .affected{{font-size:12px;color:#8b949e;font-family:"SF Mono",Menlo,monospace;margin-top:4px}}
  .badge{{display:inline-block;background:#21262d;color:#8b949e;border-radius:10px;padding:1px 8px;font-size:11px;font-weight:500;margin-left:6px;vertical-align:middle}}
  .ok{{color:#3fb950;font-size:13px;padding:4px 0}}
</style>
</head>
<body>
<header>
  <h1>🔍 DataScout IQA Healthcheck</h1>
  <div class="meta">Generated {now_str} &nbsp;·&nbsp; {n_envs} environments &nbsp;·&nbsp; {n_iqas} IQAs &nbsp;·&nbsp; Golden record: {GOLDEN_RECORD}</div>
</header>
{section("🔴", "Section 1 — Environments Down", "#f85149", s1)}
{section("🔴", "Section 2 — Broken IQAs", "#f85149", s2)}
{section("🟡", "Section 3 — IQA Issues", "#e3b341", s3)}
</body>
</html>"""

# ── Save & open ────────────────────────────────────────────────────────────
_now      = _dt.datetime.now()
_now_str  = _now.strftime("%Y-%m-%d %H:%M")
_filename = _now.strftime("healthcheck_%Y%m%d_%H%M%S.html")

_reports_dir = Path.cwd().resolve().parents[0] / "reports"
_reports_dir.mkdir(exist_ok=True)

_report_path = _reports_dir / _filename
_report_path.write_text(_build_report(_now_str), encoding="utf-8")

print(f"✅ Report saved → {_report_path}")
webbrowser.open(_report_path.as_uri())

✅ Report saved → /Users/andredowbor/Documents/datascout healthckeck use/reports/healthcheck_20260330_150959.html


True